# Конвейер обработки документов базы знаний (Docling + PyMuPDF)
### Профилирование производительности, мониторинг RAM/VRAM и адаптация под Colab / Kaggle GPU

Данный ноутбук готов к запуску **в один клик** как локально, так и в облачных средах **Google Colab** или **Kaggle** с GPU-ускорением (T4 / V100 / A100).

### Архитектурные особенности:
1. **Сквозное профилирование производительности (`profiler.py`):** замер времени каждого этапа, расчет долей в %, мониторинг пикового RAM (RSS) и VRAM GPU.
2. **Два режима обработки (`mode`):** `accurate` (полный TableFormer + OCR) и `fast` (отключение OCR при наличии печатного слоя > 100 симв/стр для резкого ускорения).
3. **Контроль утечек памяти (OOM protection):** принудительный сборщик мусора `gc.collect()` и очистка CUDA кэша `torch.cuda.empty_cache()` после каждого документа.
4. **Header Propagation & Dual Table Representation:** разворачивание `rowspan` и генерация чистого Markdown (`table_md`) + поисковых плоских фактов.
5. **Иерархический чанкинг `bge-m3`:** нарезка до 350 токенов, слияние коротких остатков < 80 токенов, признак `has_table`.
6. **Атомарный экспорт:** согласованная генерация `parsed_documents.json`, `parsed_nodes.json`, `parsed_chunks.json`, `profiling_report.json`.

In [ ]:
# [ЯЧЕЙКА 1] Определение среды, проверка GPU и установка зависимостей
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/working')

print(f"Среда исполнения: {'Google Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Локальное окружение')}")

# Проверка доступности GPU через PyTorch и nvidia-smi
try:
    import torch
    cuda_avail = torch.cuda.is_available()
    print(f"CUDA доступна: {cuda_avail}")
    if cuda_avail:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
    else:
        print("Режим работы: CPU-only (без выделенного GPU)")
except Exception as e:
    print(f"Проверка GPU завершилась с информацией: {e}")

# Установка зависимостей при запуске в Colab / Kaggle
if IN_COLAB or IN_KAGGLE:
    print("\nУстановка необходимых пакетов...")
    !pip install -q pymupdf psutil tqdm transformers sentencepiece pydantic
    # Опционально docling для ускоренной конвертации
    # !pip install -q docling

# Добавляем backend в sys.path
backend_dir = Path("..").resolve()
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from src.kb.schemas import (
    ParsedDocumentSchema,
    ParsedNodeSchema,
    ParsedChunkSchema,
    CompletenessReportSchema,
)
from src.kb.tables import TableProcessor
from src.kb.chunker import HierarchicalChunker, TokenCounter
from src.kb.parser import DocumentParser
from src.kb.profiler import PipelineProfiler, DocumentProfiler

print("Все модули подсистемы kb успешно импортированы!")

In [ ]:
# [ЯЧЕЙКА 2] Загрузка тестового документа (PDF) или создание синтетического
import fitz

test_doc_path = Path("sample_regulation.pdf")

if IN_COLAB:
    from google.colab import files
    print("Вы можете загрузить собственный регламент (PDF) через форму ниже:")
    uploaded = files.upload()
    if uploaded:
        test_doc_path = Path(list(uploaded.keys())[0])
        print(f"Используется загруженный файл: {test_doc_path}")

# Если файл не был загружен извне, создаем структурированный тестовый регламент через PyMuPDF
if not test_doc_path.exists():
    print("Создание демонстрационного документа регламента закупок с таблицей...")
    doc = fitz.open()
    page1 = doc.new_page()
    p1_text = (
        "Section 1. General Provisions of Procurement\n"
        "1.1. This Regulation establishes rules for quotation sessions on the Supplier Portal.\n"
        "1.2. Participation is open to all registered legal entities and entrepreneurs.\n\n"
        "Article 2. Security Requirements for Bids\n"
        "The security deposit amount is determined based on the initial maximum contract price.\n"
        "For procurements under 600,000 rubles, security deposit is not required.\n"
    )
    page1.insert_text((50, 60), p1_text, fontsize=11)
    page2 = doc.new_page()
    p2_text = (
        "Article 3. Contract Execution Deadlines\n"
        "3.1. The contract must be signed within 5 business days after session completion.\n"
        "3.2. Failure to sign within the deadline results in bidder disqualification.\n"
    )
    page2.insert_text((50, 60), p2_text, fontsize=11)
    doc.save(str(test_doc_path))
    doc.close()
    print(f"Сгенерирован тестовый документ: {test_doc_path} ({2} страницы)")
else:
    print(f"Выбран документ для обработки: {test_doc_path}")

In [ ]:
# [ЯЧЕЙКА 3] Инициализация DocumentParser в режиме fast с профилировщиком
pipeline_profiler = PipelineProfiler()

# Выбираем режим: 'fast' для цифровых PDF (> 100 симв/стр) или 'accurate' для полного TableFormer+OCR
parser = DocumentParser(mode="fast")
print(f"DocumentParser инициализирован в режиме: {parser.mode}")

In [ ]:
# [ЯЧЕЙКА 4] Запуск конвейера с замером производительности и демонстрацией результатов
page_count = parser.get_document_page_count(test_doc_path)
doc_profiler = pipeline_profiler.start_document(doc_id=test_doc_path.stem, page_count=page_count)

# Парсинг документа
result = parser.parse_document(
    file_path=test_doc_path,
    doc_id=test_doc_path.stem,
    regime="MOS_PORTAL",
    profiler=doc_profiler,
)

if result.profiling:
    pipeline_profiler.record_document(result.profiling)
    print("\n" + pipeline_profiler.format_table(result.profiling) + "\n")

# Демонстрация Header Propagation на тестовой таблице со сложными вертикальными объединениями
sample_table = [
    ["Категория закупки", "НМЦК", "Размер обеспечения", "Срок возврата"],
    ["Котировочная сессия", "До 600 тыс. руб.", "Не требуется", "-"],
    ["", "От 600 тыс. до 3 млн руб.", "0.5%", "5 рабочих дней"],
    ["", "От 3 млн до 5 млн руб.", "1.0%", "5 рабочих дней"],
    ["Малые закупки", "До 100 тыс. руб.", "Не требуется", "-"],
]
propagated = TableProcessor.propagate_headers_from_grid(sample_table)
print("=== Демонстрация Header Propagation (Markdown table_md) ===")
print(TableProcessor.to_markdown(propagated))

print("\n=== Первые 3 нарезанных чанка (<= 350 токенов, bge-m3) ===")
for idx, c in enumerate(result.chunks[:3], 1):
    toks = parser.token_counter.count_tokens(c.text)
    print(f"\n[Чанк #{idx} - ID: {c.chunk_id}] (токенов: {toks}, has_table: {c.has_table})")
    print(f"Текст: {c.text[:140]}...")

In [ ]:
# [ЯЧЕЙКА 5] Атомарный экспорт артефактов и упаковка в ZIP-архив для скачивания
import json
import zipfile
from src.kb.cli import atomic_write_json

out_dir = Path("./parsed_kb_output")
out_dir.mkdir(parents=True, exist_ok=True)

doc_file = out_dir / "parsed_documents.json"
node_file = out_dir / "parsed_nodes.json"
chunk_file = out_dir / "parsed_chunks.json"
rep_file = out_dir / "completeness_reports.json"
prof_file = out_dir / "profiling_report.json"

# Атомарное сохранение
atomic_write_json(doc_file, [result.document.model_dump(mode="json")])
atomic_write_json(node_file, [n.model_dump(mode="json") for n in result.nodes])
atomic_write_json(chunk_file, [c.model_dump(mode="json") for c in result.chunks])
atomic_write_json(rep_file, [result.completeness_report.model_dump(mode="json")])
pipeline_profiler.save_atomic_report(prof_file)

# Упаковка в ZIP-архив
zip_path = Path("ingestion_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for json_f in (doc_file, node_file, chunk_file, rep_file, prof_file):
        zipf.write(json_f, arcname=json_f.name)

print(f"Все артефакты упакованы в архив: {zip_path.resolve()} ({zip_path.stat().st_size} байт)")

if IN_COLAB:
    from google.colab import files
    print("Инициирование скачивания архива в браузере...")
    files.download(str(zip_path))
else:
    print(f"Локальный архив готов для использования: {zip_path.absolute()}")